# Customer Churn, Cohort Analysis

Model performance numbers don't translate directly into business impact. This notebook cuts the data by contract type and tenure cohort to find which customer segments carry the most churn risk, and quantifies the monthly revenue at stake in each one.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

warnings.filterwarnings("ignore")
os.makedirs("../visuals", exist_ok=True)

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
df["Churn_bin"] = (df["Churn"] == "Yes").astype(int)
print(f"Loaded {len(df):,} customers")

Binning tenure into 6-month and yearly cohorts makes the patterns easier to read than treating tenure as a continuous variable in a heatmap.

In [ ]:
bins = [0, 6, 12, 24, 36, 48, 72]
labels = ["0-6m", "7-12m", "13-24m", "25-36m", "37-48m", "49-72m"]
df["TenureCohort"] = pd.cut(df["tenure"], bins=bins, labels=labels, right=True)
print(df["TenureCohort"].value_counts().sort_index())

Month-to-month customers in the first six months are churning at over 50% in some cases. Two-year contract customers barely show up regardless of tenure, the contract itself is doing retention work the model can't replicate.

In [ ]:
pivot = df.pivot_table(
    values="Churn_bin",
    index="Contract",
    columns="TenureCohort",
    aggfunc="mean"
) * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, annot_kws={"size": 11},
            cbar_kws={"label": "Churn Rate (%)"})
ax.set_title("Churn Rate (%) by Contract Type x Tenure Cohort",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Tenure Cohort")
ax.set_ylabel("Contract Type")
plt.tight_layout()
plt.savefig("../visuals/cohort_churn_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
rar = (
    df.groupby(["Contract", "TenureCohort"])
    .apply(lambda g: pd.Series({
        "Customers": len(g),
        "ChurnRate": g["Churn_bin"].mean() * 100,
        "AtRiskCustomers": int(g["Churn_bin"].sum()),
        "RevenueAtRisk_USD" : g.loc[g["Churn_bin"] == 1, "MonthlyCharges"].sum()
    }))
    .reset_index()
)

print(rar.sort_values("RevenueAtRisk_USD", ascending=False).to_string(index=False))

High churn rate does not always mean high revenue at risk, a segment with 50% churn but few customers costs less than one with 25% churn across a large base. The revenue heatmap makes this trade-off explicit.

In [ ]:
pivot_rar = rar.pivot_table(
    values="RevenueAtRisk_USD",
    index="Contract",
    columns="TenureCohort",
    aggfunc="sum"
).fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot_rar / 1000, annot=True, fmt=".1f", cmap="Reds",
            linewidths=0.5, ax=ax, annot_kws={"size": 11},
            cbar_kws={"label": "Revenue at Risk ($K/month)"})
ax.set_title("Revenue at Risk ($K/month) by Contract and Tenure Cohort",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Tenure Cohort")
ax.set_ylabel("Contract Type")
plt.tight_layout()
plt.savefig("../visuals/revenue_at_risk_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
cohort_churn = df.groupby("TenureCohort")["Churn_bin"].mean() * 100
contract_rar = rar.groupby("Contract")["RevenueAtRisk_USD"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cohort_churn.index.astype(str), cohort_churn.values, color="#DD8452", edgecolor="white")
for i, v in enumerate(cohort_churn.values):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)
axes[0].set_title("Churn Rate by Tenure Cohort", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Churn Rate (%)")

axes[1].bar(contract_rar.index, contract_rar.values / 1000, color="#4C72B0", edgecolor="white")
for i, v in enumerate(contract_rar.values / 1000):
    axes[1].text(i, v + 0.3, f"${v:.1f}K", ha="center", fontsize=9)
axes[1].set_title("Revenue at Risk by Contract Type", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Revenue at Risk ($K/month)")

plt.tight_layout()
plt.savefig("../visuals/cohort_summary.png", bbox_inches="tight")
plt.show()

In [ ]:
total_rar = rar["RevenueAtRisk_USD"].sum()
top_segment = rar.sort_values("RevenueAtRisk_USD", ascending=False).iloc[0]

print(f"Total revenue at risk/month : ${total_rar:,.0f}")
print(f"Highest risk segment        : {top_segment['Contract']} / {top_segment['TenureCohort']}")
print(f"  Churn rate                : {top_segment['ChurnRate']:.1f}%")
print(f"  Revenue at risk           : ${top_segment['RevenueAtRisk_USD']:,.0f}/month")